# 02 — Liver Steatosis Binary Model (Proof of Concept)Binary classification: can Whole Blood expression predict liver steatosis?

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom collections import Counterfrom sklearn.metrics import roc_curve, roc_auc_score, mean_absolute_errorfrom gtex_biomarkers.config import Configfrom gtex_biomarkers.data import (    load_raw_data, filter_whole_blood,    build_blood_expression_matrix, build_blood_subjid,)from gtex_biomarkers.models import run_cv, make_lr_pipelinefrom gtex_biomarkers.evaluation import plot_roc_foldsConfig.ensure_dirs()# Load data (or reuse from notebook 01 if running in same session)df_expr, df_samples, df_age, df_meta_url = load_raw_data()blood_meta = filter_whole_blood(df_samples)X_wb, df_blood_wb = build_blood_expression_matrix(df_expr, blood_meta)blood_subjid = build_blood_subjid(X_wb)

## Create Steatosis Label from Pathology Metadata

In [ ]:
liver_path = df_meta_url.loc[    df_meta_url["Tissue"].astype(str) == "Liver",    ["Tissue.Sample.ID", "Tissue", "Pathology.Categories", "Pathology.Notes"]].copy()liver_path = liver_path.rename(columns={"Tissue.Sample.ID": "SAMPID"})text = (    liver_path["Pathology.Categories"].fillna("").astype(str) + " " +    liver_path["Pathology.Notes"].fillna("").astype(str))liver_path["steatosis_label"] = text.str.contains("steatosis", case=False).astype(int)liver_path["SUBJID"] = liver_path["SAMPID"].astype(str).str.split("-").str[:2].str.join("-")donor_steatosis = liver_path.groupby("SUBJID")["steatosis_label"].max()print("Donors with steatosis label:")print(donor_steatosis.value_counts())

## Build Labeled Blood Dataset

In [ ]:
y = blood_subjid.map(donor_steatosis)keep = y.notna()X_features = X_wb.loc[keep].copy()y = y.loc[keep].astype(int)groups = blood_subjid.loc[keep].astype(str)print(f"Labeled blood samples: {X_features.shape[0]}")print(f"Features (genes): {X_features.shape[1]}")print(f"Label distribution:\n{y.value_counts()}")

## Run 5-Fold Grouped CV

In [ ]:
results = run_cv(X_features, y, groups, make_lr_pipeline)print(f"Mean AUC: {results['mean_auc']:.3f} ± {results['std_auc']:.3f}")for i, a in enumerate(results['fold_aucs'], 1):    print(f"  Fold {i}: AUC = {a:.3f}")

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))plot_roc_folds(results, title="Liver Steatosis (Binary)", ax=ax)fig.savefig(Config.FIGURES_DIR / "roc_curves_5fold.pdf", bbox_inches="tight")plt.show()